# Aula 24 — Explainability, Fairness e LLMs

Laboratório CPU-only: SHAP/LIME, métricas de fairness e retrieval RAG local.

In [ ]:
%pip -q install 'numpy>=1.24,<3' 'pandas>=2,<3' 'scikit-learn>=1.3,<2' 'matplotlib>=3.7,<4' 'joblib>=1.3,<2' 'shap>=0.46,<1' 'lime>=0.2,<1' 'fairlearn>=0.11,<1'

In [ ]:
from pathlib import Path
import json
import os
import sys

ACTIVITY = Path.cwd()
if not (ACTIVITY / 'src' / 'train_model.py').exists():
    candidates = list(Path('/content').glob('**/aula24/monitor/atividade'))
    if candidates:
        ACTIVITY = candidates[0]
os.chdir(ACTIVITY)
sys.path.insert(0, str(ACTIVITY))
assert (ACTIVITY / 'src' / 'train_model.py').exists(), 'Abra o notebook a partir da pasta atividade.'
print('Atividade:', ACTIVITY)

## 1. Dados e treinamento

O atributo `protected_group` será preservado para auditoria, mas não poderá entrar nas features do modelo.

In [ ]:
from src.common import make_credit_dataset, save_dataset
from src.train_model import train

save_dataset(make_credit_dataset(seed=42))
metrics = train(seed=42)
metrics

## 2. SHAP e LIME

Compare a explicação global do SHAP e as explicações locais de SHAP/LIME para a mesma linha.

In [ ]:
from src.explain import explain_shap, explain_lime

shap_files = explain_shap(row_index=0)
lime_files = explain_lime(row_index=0)
shap_files, lime_files

In [ ]:
from IPython.display import Image, display
display(Image(filename=shap_files['global_plot']))
display(Image(filename=shap_files['local_plot']))
print('Abra o HTML do LIME em:', lime_files['html'])

## 3. Fairness por grupo

Nenhuma métrica isolada define fairness. Observe as diferenças entre selection rate, TPR e FPR.

In [ ]:
from src.fairness import assess
fairness_result = assess()
print(json.dumps(fairness_result, indent=2, ensure_ascii=False))

## 4. Retrieval RAG local

O retrieval recupera fontes e monta o contexto; não há geração externa nesta atividade.

In [ ]:
from src.rag_retrieve import retrieve

rag_result = retrieve('quando usar RAG em vez de fine-tuning?', top_k=3)
for hit in rag_result['hits']:
    print(f"[{hit['source']}] score={hit['score']:.3f} — {hit['title']}")
print('\nPROMPT MONTADO:\n')
print(rag_result['prompt'])

## Discussão

1. A explicação SHAP/LIME é causal?
2. O que a `area_proxy` revela sobre a remoção do atributo protegido?
3. Qual critério de fairness é adequado ao caso de crédito?
4. Por que atualizar o índice RAG pode ser melhor que fazer fine-tuning?
5. Que evidências devem ser registradas para auditar a resposta de um LLM?